# Reefer Load Outlook Challenge

## Problem Description
The objective of this challenge is to forecast the combined electricity consumption of **reefer containers** (refrigerated shipping containers) plugged in at a terminal for specific future hourly timestamps.

### Why is this important?
Reefer containers require constant power to maintain their internal temperature. At large terminals, hundreds or thousands of containers can be plugged in simultaneously, creating significant electricity demand. Predicting this demand helps terminal operators:
- Manage energy costs.
- Coordinate with the power grid.
- Avoid peak load surcharges.

### The Task
For each timestamp in `target_timestamps.csv`, we need to provide:
1. **`pred_power_kw`**: A point forecast of the total power consumption (kW).
2. **`pred_p90_kw`**: A cautious upper estimate (90th percentile), representing a "peak" scenario.

### Data Sources
- **Reefer Release Data**: Historical power consumption and container details.
- **Weather Data**: Industrial temperature and wind measurements, which influence cooling demand.

### Baseline Strategy
A simple baseline approach uses the load from the same hour one day earlier (`t-24h`) and adds a 10% safety margin for the P90 estimate.

# Reefer Challenge Starter Notebook

This notebook builds a very simple valid public leaderboard submission for the hackathon challenge.

What it does:
- reads the released public `target_timestamps.csv`
- aggregates hourly reefer power consumption from `reefer_release.zip`
- uses a naive `t-24h` forecast
- adds a simple `pred_p90_kw = 1.10 * pred_power_kw`
- writes `predictions.csv` for the public leaderboard target hours

This is only a starter. Teams should improve it with better feature engineering, validation, and uncertainty handling.

In [1]:
from __future__ import annotations
import pandas as pd
import numpy as np
import zipfile
import io
from datetime import datetime, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

def find_public_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "participant_package" / "participant_package",
        Path.cwd() / "challenge" / "release" / "v1" / "public",
        Path.cwd() / "challenge" / "bundle" / "v1" / "participant_package",
        Path.cwd().parent,
        Path.cwd().parent / "public",
    ]
    for candidate in candidates:
        if (candidate / "reefer_release.zip").exists() and (candidate / "target_timestamps.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate the participant data directory.")

PUBLIC_DIR = find_public_dir()
REEFER_ZIP = PUBLIC_DIR / "reefer_release.zip"
TARGETS_CSV = PUBLIC_DIR / "target_timestamps.csv"
WEATHER_DIR = PUBLIC_DIR / "wetterdaten" / "Wetterdaten Okt 25 - 23 Feb 26"
OUTPUT_DIR = PUBLIC_DIR / "starter" / "output" if (PUBLIC_DIR / "starter").exists() else PUBLIC_DIR / "output"
OUTPUT_CSV = OUTPUT_DIR / "predictions.csv"

# Verify paths
print(f"Data Dir: {PUBLIC_DIR}")
print(f"Reefer ZIP exists: {REEFER_ZIP.exists()}")
print(f"Targets CSV exists: {TARGETS_CSV.exists()}")
print(f"Weather Dir exists: {WEATHER_DIR.exists()}")

(PosixPath('/Users/wegel/Documents/Projects/Hackathon 2026/participant_package'),
 True,
 True)

In [ ]:
def load_weather_data(weather_dir: Path) -> pd.DataFrame:
    temp_file = weather_dir / "CTH_Temperatur_VC_Halle3 Okt 25 - 23 Feb 26.csv"
    df = pd.read_csv(temp_file, sep=";", decimal=",", parse_dates=["UtcTimestamp"])
    
    # Aggregate to hourly (mean temperature)
    df['timestamp_utc'] = df['UtcTimestamp'].dt.floor('h')
    weather_hourly = df.groupby('timestamp_utc')['Value'].mean().reset_index()
    weather_hourly.rename(columns={'Value': 'temp_c'}, inplace=True)
    return weather_hourly

weather_df = load_weather_data(WEATHER_DIR)
print(f"Loaded {len(weather_df)} weather records.")
weather_df.head()

In [2]:
def parse_decimal(value: str) -> float | None:
    text = value.strip()
    if not text:
        return None
    try:
        return float(text.replace(",", "."))
    except ValueError:
        return None


def parse_timestamp(value: str) -> datetime:
    text = value.strip()
    if text.endswith("Z"):
        text = text[:-1]
    return datetime.fromisoformat(text.replace(" ", "T"))


def to_hour(ts: datetime) -> datetime:
    return ts.replace(minute=0, second=0, microsecond=0)


def iso_utc(ts: datetime) -> str:
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


In [3]:
def load_reefer_data(reefer_zip: Path) -> pd.DataFrame:
    with zipfile.ZipFile(reefer_zip) as zf:
        with zf.open("reefer_release.csv") as raw:
            # Note: Pandas can read directly from zip binary stream
            df = pd.read_csv(raw, sep=";", decimal=",", parse_dates=["EventTime"], encoding="utf-8-sig")
    
    # Aggregate power (W -> kW) per hour
    df['timestamp_utc'] = df['EventTime'].dt.floor('h')
    # Filter rows with missing Power
    df = df.dropna(subset=['AvPowerCons'])
    
    hourly_load = df.groupby('timestamp_utc')['AvPowerCons'].sum() / 1000.0
    return hourly_load.reset_index(name='power_kw')

load_df = load_reefer_data(REEFER_ZIP)
print(f"Loaded {len(load_df)} hourly load records.")
load_df.head()

(223, 8403)

In [ ]:
def create_features(df, weather_df=None):
    df = df.copy()
    df['hour'] = df['timestamp_utc'].dt.hour
    df['day_of_week'] = df['timestamp_utc'].dt.dayofweek
    
    # Merge weather if provided
    if weather_df is not None:
        df = pd.merge(df, weather_df, on='timestamp_utc', how='left')
        # Fill missing weather with mean or forward fill
        df['temp_c'] = df['temp_c'].ffill().bfill()
        
    # Lag features (only if 'power_kw' exists, i.e., training data)
    # For inference, we need to pass the lags separately or have them in the df
    if 'power_kw' in df.columns:
        df['power_kw_lag24'] = df['power_kw'].shift(24)
        df['power_kw_lag168'] = df['power_kw'].shift(168)
        
    return df

# Create training dataset
train_df = pd.merge(load_df, weather_df, on='timestamp_utc', how='left')
train_df['temp_c'] = train_df['temp_c'].ffill().bfill()
train_df['hour'] = train_df['timestamp_utc'].dt.hour
train_df['day_of_week'] = train_df['timestamp_utc'].dt.dayofweek
train_df['power_kw_lag24'] = train_df['power_kw'].shift(24)
train_df['power_kw_lag168'] = train_df['power_kw'].shift(168)

# Drop rows with NaN lags
train_df = train_df.dropna().reset_index(drop=True)
print(f"Training features created. Shape: {train_df.shape}")
train_df.head()

In [ ]:
# Simple Time-Series Split
split_date = train_df['timestamp_utc'].max() - timedelta(days=7)
X = train_df[['hour', 'day_of_week', 'temp_c', 'power_kw_lag24', 'power_kw_lag168']]
y = train_df['power_kw']

mask_train = train_df['timestamp_utc'] <= split_date
X_train, y_train = X[mask_train], y[mask_train]
X_val, y_val = X[~mask_train], y[~mask_train]

model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
mae = mean_absolute_error(y_val, y_pred)
print(f"Validation MAE: {mae:.2f} kW")

# Visual comparison
plt.figure(figsize=(15, 5))
plt.plot(train_df.loc[~mask_train, 'timestamp_utc'], y_val, label='Actual')
plt.plot(train_df.loc[~mask_train, 'timestamp_utc'], y_pred, label='XGBoost Prediction')
plt.legend()
plt.title("Validation Set - Actual vs Predicted")
plt.show()

In [4]:
# Prepare target hours for prediction
targets_df = pd.read_csv(TARGETS_CSV, parse_dates=['timestamp_utc'])
targets_df['timestamp_utc'] = targets_df['timestamp_utc'].dt.tz_localize(None) # Remove Z for merging

# We need the most recent lags for prediction
full_history = load_df.set_index('timestamp_utc')['power_kw']

results = []
for idx, row in targets_df.iterrows():
    ts = row['timestamp_utc']
    
    # Get weather
    temp_row = weather_df[weather_df['timestamp_utc'] == ts]
    temp = temp_row['temp_c'].values[0] if not temp_row.empty else weather_df['temp_c'].mean()
    
    # Get lags (we use the actual values from history)
    lag24 = full_history.get(ts - timedelta(hours=24), 0)
    lag168 = full_history.get(ts - timedelta(hours=168), 0)
    
    # Features for model
    feat = pd.DataFrame([[ts.hour, ts.dayofweek, temp, lag24, lag168]], 
                        columns=['hour', 'day_of_week', 'temp_c', 'power_kw_lag24', 'power_kw_lag168'])
    
    pred_power = model.predict(feat)[0]
    
    # Improved p90: let's use a 15% safety margin or calibrate based on validation error
    # Here we stick to a slightly better heuristic for now
    pred_p90 = pred_power * 1.15 if pred_power > 0 else 0
    
    results.append({
        'timestamp_utc': ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
        'pred_power_kw': round(float(pred_power), 6),
        'pred_p90_kw': round(float(pred_p90), 6)
    })

predictions = results
print(f"Generated {len(predictions)} predictions.")
predictions[:3]

[{'timestamp_utc': '2026-01-01T00:00:00Z',
  'pred_power_kw': 834.463337,
  'pred_p90_kw': 917.909671},
 {'timestamp_utc': '2026-01-01T01:00:00Z',
  'pred_power_kw': 826.299683,
  'pred_p90_kw': 908.929651},
 {'timestamp_utc': '2026-01-01T02:00:00Z',
  'pred_power_kw': 833.958823,
  'pred_p90_kw': 917.354705}]

In [5]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["timestamp_utc", "pred_power_kw", "pred_p90_kw"])
    writer.writeheader()
    writer.writerows(predictions)

OUTPUT_CSV

PosixPath('/Users/wegel/Documents/Projects/Hackathon 2026/participant_package/starter/output/predictions.csv')

## Next improvements

Good next steps for participants:
- add weather lags from `wetterdaten.zip`
- validate with forward-only time splits
- model peaks separately
- calibrate `pred_p90_kw` on a holdout period instead of using a fixed `10%` uplift

Organizer note: for final scoring, rerun the handed-in model/code on a hidden full target list that also includes the private timestamps.